# 07 音高、和弦与调性

对应正文单声部/多声部音高、模板和弦、RMVPE 与 K-S 调性估计部分。

1. pYIN 与 torchcrepe：同一单声部片段的无真值输出对照
2. MultiPitchMelodia 与 Basic Pitch：不同粒度的多音高输出
3. 模板和弦与 Essentia `ChordsDetection`
4. RMVPE 社区 ONNX 封装：同一模型在混音与纯人声分轨上的输出对照
5. Krumhansl--Schmuckler profile 排名


## 1. 环境自检与依赖说明

Notebook 只检查依赖，不在执行过程中自动 `pip install`。若当前内核与 Basic Pitch 的依赖不兼容，可把环境变量 `BASIC_PITCH_PYTHON` 指向另一个已安装 Basic Pitch 的 Python 解释器。


In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from IPython import display as ipydisplay
from pathlib import Path

import librosa
import librosa.display

SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)
OUTPUT_AUDIO_DIR = BASE_DIR / "CODE" / "chapter05" / "output_audio"
OUTPUT_AUDIO_DIR.mkdir(exist_ok=True)

import importlib.metadata
print(f"librosa={importlib.metadata.version('librosa')}")


In [ ]:
# 依赖状态检查；不自动安装
import os
import json
import subprocess
import tempfile

for package_name, import_name in [
    ("torchcrepe", "torchcrepe"),
    ("basic-pitch", "basic_pitch"),
    ("rmvpe-onnx", "rmvpe_onnx"),
    ("essentia", "essentia"),
]:
    try:
        module = __import__(import_name)
        version = getattr(module, "__version__", "installed")
        print(f"{package_name}: {version}")
    except Exception as exc:
        print(f"{package_name}: 当前内核不可用 ({type(exc).__name__})")

BASIC_PITCH_PYTHON = os.environ.get("BASIC_PITCH_PYTHON")
if BASIC_PITCH_PYTHON:
    print(f"Basic Pitch 外部解释器：{BASIC_PITCH_PYTHON}")


## 2. 单声部音高估计：pYIN 与 torchcrepe

pYIN 把周期性、阈值分布与 HMM 连续性写进算法；torchcrepe 使用 CREPE CNN 权重，并默认采用 Viterbi 解码。

In [ ]:
violin_samples, sr = librosa.load(
    DATASET_DIR / "zhao_violin_dry.wav", sr=SAMPLE_RATE, mono=True, offset=15.0, duration=5.0
)
print(f"加载小提琴干声：{len(violin_samples)} samples ({len(violin_samples)/sr:.2f} s)")
display(ipydisplay.Audio(violin_samples, rate=SAMPLE_RATE))

In [ ]:
# pYIN：非学习型方法，基于 YIN 差分函数的周期性证据、阈值概率模型与 HMM
f0_pyin, voiced_flag, voiced_probs = librosa.pyin(
    violin_samples,
    fmin=librosa.note_to_hz("G3"),
    fmax=librosa.note_to_hz("G6"),
    sr=SAMPLE_RATE
)
time_pyin = librosa.times_like(f0_pyin, sr=SAMPLE_RATE)

print(f"pYIN 输出：{len(f0_pyin)} 帧，voiced 比例 = {np.mean(voiced_flag):.2%}")
valid_pyin = f0_pyin[np.isfinite(f0_pyin)]
if valid_pyin.size:
    print(f"音高范围：{valid_pyin.min():.1f} – {valid_pyin.max():.1f} Hz")
else:
    print("音高范围：没有有限的 voiced F0 输出")

In [ ]:
# torchcrepe：CREPE CNN + 默认 Viterbi 解码
torchcrepe_available = False
try:
    import torch
    import torchcrepe as tc
    torchcrepe_available = True
    print("torchcrepe 已加载；predict 默认使用 Viterbi decoder")
except ImportError:
    print("torchcrepe 未安装，跳过该输出。")
    tc = None


In [ ]:
f0_crepe = confidence = time_crepe = None

if torchcrepe_available:
    audio_tensor = torch.from_numpy(violin_samples).unsqueeze(0).float()
    # torchcrepe 会在离散音高 bin 上加入随机音分抖动；替换再恢复 NumPy 状态以稳定图表。
    dither_rng_state = np.random.get_state()
    np.random.seed(202605)
    try:
        f0_crepe, confidence = tc.predict(
            audio_tensor,
            SAMPLE_RATE,
            hop_length=256,
            fmin=librosa.note_to_hz("G3"),
            fmax=librosa.note_to_hz("G6"),
            model="tiny",
            return_periodicity=True,
            device="cpu",
        )
    finally:
        np.random.set_state(dither_rng_state)
    f0_crepe = f0_crepe.squeeze().cpu().numpy()
    confidence = confidence.squeeze().cpu().numpy()
    time_crepe = librosa.times_like(f0_crepe, sr=SAMPLE_RATE, hop_length=256)

    print(f"torchcrepe 输出：{len(f0_crepe)} 帧")
    valid_crepe = f0_crepe[np.isfinite(f0_crepe) & (f0_crepe > 0)]
    if valid_crepe.size:
        print(f"音高范围：{valid_crepe.min():.1f} – {valid_crepe.max():.1f} Hz")
    else:
        print("音高范围：没有有限的正频率输出")
    print(f"平均 periodicity = {np.mean(confidence):.3f}")
else:
    print("跳过 torchcrepe 推理")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# 上：pYIN 输出
axes[0].plot(time_pyin, f0_pyin, "o-", color="0.5", markersize=2, lw=0.8, label="pYIN")
axes[0].set_xlabel("")
axes[0].set_ylabel("频率 (Hz)")
axes[0].set_title("单声部音高：pYIN")
axes[0].legend()
axes[0].set_ylim(420, 650)

# 下：torchcrepe 与 pYIN 输出（如有）
if torchcrepe_available:
    # 只画置信度 > 0.5 的点
    mask = confidence > 0.5
    axes[1].plot(time_crepe[mask], f0_crepe[mask], "s-", color="0.3", markersize=2, lw=0.8, label="torchcrepe（置信度>0.5）")
    axes[1].plot(time_pyin, f0_pyin, "o-", color="0.6", markersize=1, lw=0.5, alpha=0.5, label="pYIN")
    axes[1].set_title("单声部音高：torchcrepe 与 pYIN")
else:
    axes[1].plot(time_pyin, f0_pyin, "o-", color="0.6", markersize=2, lw=0.8, label="pYIN")
    axes[1].set_title("单声部音高：仅 pYIN（torchcrepe 未安装）")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("频率 (Hz)")
axes[1].legend()
axes[1].set_ylim(420, 650)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "pitch_pyin_vs_crepe.png", dpi=600, bbox_inches="tight")
plt.show()

## 3. 多声部音高：MultiPitchMelodia 与 Basic Pitch

`MultiPitchMelodia` 是 Essentia 中基于 pitch contours 的多声源适配；不能简单说成“把 PredominantPitchMelodia 最后一步删掉”。Basic Pitch 有 16,782 个参数，输出 `contour`、`onset`、`note` 三张帧级激活图，再生成含 amplitude 与可选 pitch bends 的 note events。

Basic Pitch 官方将模型描述为跨乐器通用，但提示一次处理一种乐器通常最合适。

In [ ]:
# Basic Pitch 可在当前内核运行，也可由 BASIC_PITCH_PYTHON 指向兼容解释器
basic_pitch_mode = None
bp_predict = None
try:
    from basic_pitch.inference import predict as bp_predict
    basic_pitch_mode = "in_process"
    print("Basic Pitch：当前内核可用")
except Exception as exc:
    if BASIC_PITCH_PYTHON and Path(BASIC_PITCH_PYTHON).exists():
        basic_pitch_mode = "external"
        print(f"Basic Pitch：将使用外部解释器 ({type(exc).__name__})")
    else:
        print("Basic Pitch 不可用；设置 BASIC_PITCH_PYTHON 可启用外部兼容环境。")


In [ ]:
# 加载钢琴片段
piano_samples, sr = librosa.load(
    DATASET_DIR / "piano_solo.wav", sr=SAMPLE_RATE, mono=True, offset=10.0, duration=5.0)

# 播放
ipydisplay.display(ipydisplay.Audio(data=piano_samples, rate=SAMPLE_RATE))

In [ ]:
# MultiPitchMelodia（Essentia）：多声部音高估计

import essentia.standard as es

mpm = es.MultiPitchMelodia(sampleRate=SAMPLE_RATE, frameSize=2048, hopSize=512)  # type: ignore[operator]
pitches = mpm(piano_samples)

time_mpm = np.arange(len(pitches)) * 512 / SAMPLE_RATE

# 收集音高事件并转为 MIDI 音高
times_all, midi_all, colors_all = [], [], []
for i, frame_pitches in enumerate(pitches):
    t = time_mpm[i]
    for f in frame_pitches:
        if f > 0:
            midi = librosa.hz_to_midi(float(f))
            if midi > 0:
                times_all.append(t)
                midi_all.append(midi)
                colors_all.append(0.3 + 0.35 * (len(midi_all) % 3))

fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(times_all, midi_all, c=colors_all, s=10, alpha=0.7, cmap="Greys", vmin=0, vmax=1)

# MIDI 音高网格：每 12 个半音一条主网格线
midi_min = int(np.floor(min(midi_all) / 12) * 12) if midi_all else 48
midi_max = int(np.ceil(max(midi_all) / 12) * 12) if midi_all else 84
for m in range(midi_min, midi_max + 1, 12):
    ax.axhline(m, color="0.8", linewidth=0.4, linestyle="-")

ax.set_xlabel("时间 (s)")
ax.set_ylabel("MIDI 音高")
yticks = list(range(midi_min, midi_max + 1, 12))
ax.set_yticks(yticks)
ax.set_yticklabels([librosa.midi_to_note(m) for m in yticks])
ax.set_ylim(midi_min - 2, midi_max + 2)
ax.set_title("MultiPitchMelodia（Essentia）：钢琴多声部音高估计（MIDI 坐标）", loc="left")
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "multipitch_melodia_piano.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
# Basic Pitch：同时展示 contour 激活图与 note-event piano roll
basic_pitch_results_available = False
model_output = None
note_events = []

if basic_pitch_mode:
    import soundfile as sf

    temp_piano = OUTPUT_AUDIO_DIR / "_temp_basic_pitch_piano.wav"
    sf.write(temp_piano, piano_samples, SAMPLE_RATE, subtype="PCM_16")

    if basic_pitch_mode == "in_process":
        model_output, _, note_events = bp_predict(str(temp_piano))
    else:
        helper_code = r"""
import json
import sys
import numpy as np
from basic_pitch.inference import predict

model_output, _, note_events = predict(sys.argv[1])
np.savez_compressed(
    sys.argv[2],
    contour=model_output["contour"],
    onset=model_output["onset"],
    note=model_output["note"],
)
serial = []
for start, end, pitch, amplitude, bends in note_events:
    serial.append([
        float(start), float(end), int(pitch), float(amplitude),
        None if bends is None else [int(value) for value in bends],
    ])
with open(sys.argv[3], "w", encoding="utf-8") as handle:
    json.dump(serial, handle, ensure_ascii=False)
"""
        with tempfile.TemporaryDirectory(prefix="chapter05_basic_pitch_") as bp_tmp:
            bp_tmp_dir = Path(bp_tmp)
            npz_path = bp_tmp_dir / "basic_pitch_output.npz"
            json_path = bp_tmp_dir / "basic_pitch_events.json"
            child_env = os.environ.copy()
            child_env["NUMBA_CACHE_DIR"] = str(bp_tmp_dir.resolve())
            child_env["TMPDIR"] = str(bp_tmp_dir.resolve())
            completed = subprocess.run(
                [BASIC_PITCH_PYTHON, "-c", helper_code,
                 str(temp_piano), str(npz_path), str(json_path)],
                check=False, env=child_env, capture_output=True, text=True,
            )
            if completed.returncode != 0:
                detail = completed.stderr.strip() or completed.stdout.strip()
                raise RuntimeError("Basic Pitch 外部推理失败：\n" + detail[-4000:])
            with np.load(npz_path) as arrays:
                model_output = {key: arrays[key] for key in ["contour", "onset", "note"]}
            note_events = json.loads(json_path.read_text(encoding="utf-8"))

    basic_pitch_results_available = True
    print("激活图形状：", {key: value.shape for key, value in model_output.items()})
    print(f"音符事件：{len(note_events)} 个")
    for event in note_events[:5]:
        print(
            f"  {event[0]:.3f}s–{event[1]:.3f}s | MIDI {int(event[2])} | "
            f"amplitude={event[3]:.3f} | bends={'yes' if event[4] else 'no'}"
        )

    duration = len(piano_samples) / SAMPLE_RATE
    contour = model_output["contour"]
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    contour_img = axes[0].imshow(
        contour.T, origin="lower", aspect="auto", cmap="Greys_r",
        extent=[0, duration, 21, 109], vmin=0, vmax=1,
    )
    axes[0].set_ylabel("细分音高网格（3 bin/半音）")
    axes[0].set_title("Basic Pitch contour 激活图（264 个 bin）", loc="left")
    fig.colorbar(contour_img, ax=axes[0], label="激活值")

    for start, end, pitch, amplitude, _ in note_events:
        axes[1].barh(
            int(pitch), width=float(end) - float(start), left=float(start),
            height=0.7, color="0.25", alpha=float(np.clip(amplitude, 0.15, 1.0)),
        )
    axes[1].set_xlabel("时间 (s)")
    axes[1].set_ylabel("MIDI 音高")
    axes[1].set_title("后处理音符事件（含 amplitude；pitch bend 未展开）", loc="left")
    axes[1].set_xlim(0, duration)

    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_DIR / "basic_pitch_piano_roll.png", dpi=600, bbox_inches="tight")
    plt.show()
    temp_piano.unlink(missing_ok=True)
else:
    print("未运行 Basic Pitch；后续保留 MultiPitchMelodia 结果。")


## 4. 模板法和弦识别

24 个 major/minor 模板不能表示所有和弦，还必须显式处理低能量/静音帧。下面先沿时间对每一类相似度做中值滤波，再 argmax；不能直接对无序的标签整数做中值滤波。


In [ ]:
def generate_chord_templates():
    """生成 24 个 L2 归一化 major/minor chroma 模板。"""
    templates = {}
    chroma_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    for root in range(12):
        major = np.zeros(12)
        major[(root + np.array([0, 4, 7])) % 12] = 1.0
        templates[f"{chroma_names[root]}:maj"] = major / np.linalg.norm(major)

        minor = np.zeros(12)
        minor[(root + np.array([0, 3, 7])) % 12] = 1.0
        templates[f"{chroma_names[root]}:min"] = minor / np.linalg.norm(minor)
    return templates

def chord_recognition_template(chroma, rms, templates, smooth_size=5, silence_ratio=0.02):
    """先平滑每类相似度，再分类；低 RMS 或零范数 chroma 帧输出 N。"""
    from scipy.ndimage import median_filter

    template_names = list(templates)
    template_matrix = np.array([templates[name] for name in template_names])
    chroma_magnitude = np.linalg.norm(chroma, axis=0)
    valid_chroma = chroma_magnitude > 0
    chroma_norm = np.divide(
        chroma, chroma_magnitude[None, :],
        out=np.zeros_like(chroma),
        where=valid_chroma[None, :],
    )
    similarities = template_matrix @ chroma_norm
    smoothed_similarities = median_filter(
        similarities, size=(1, smooth_size), mode="nearest"
    )

    n_frames = min(chroma.shape[1], len(rms))
    similarities = similarities[:, :n_frames]
    smoothed_similarities = smoothed_similarities[:, :n_frames]
    raw_idx = np.argmax(similarities, axis=0)
    best_idx = np.argmax(smoothed_similarities, axis=0)
    best_scores = np.max(smoothed_similarities, axis=0)

    silence_threshold = silence_ratio * np.max(rms) if np.max(rms) > 0 else 0.0
    no_chord = (rms[:n_frames] <= silence_threshold) | ~valid_chroma[:n_frames]
    raw_labels = [
        "N" if no_chord[t] else template_names[index]
        for t, index in enumerate(raw_idx)
    ]
    labels = [
        "N" if no_chord[t] else template_names[index]
        for t, index in enumerate(best_idx)
    ]
    return raw_labels, labels, best_scores, similarities, smoothed_similarities, no_chord

chord_templates = generate_chord_templates()
print(f"已生成 {len(chord_templates)} 个和弦模板；另用能量门限输出 N")


In [ ]:
# 重新加载钢琴片段，计算 chroma 与同帧率 RMS
piano_samples, sr = librosa.load(
    DATASET_DIR / "piano_solo.wav", sr=SAMPLE_RATE,
    mono=True, offset=12.0, duration=3.0
)
chroma = librosa.feature.chroma_cqt(
    y=piano_samples, sr=SAMPLE_RATE, hop_length=512
)
rms_chord = librosa.feature.rms(
    y=piano_samples, frame_length=2048, hop_length=512
)[0]

raw_chords, smoothed_chords, scores, raw_sims, all_sims, no_chord = chord_recognition_template(
    chroma, rms_chord, chord_templates
)
n_chord_frames = len(smoothed_chords)
chroma = chroma[:, :n_chord_frames]
time_chords = librosa.frames_to_time(
    np.arange(n_chord_frames), sr=SAMPLE_RATE, hop_length=512
)

print(f"Chroma 形状：{chroma.shape}")
print(f"N 帧：{np.sum(no_chord)} / {n_chord_frames}")
print(f"原始/平滑标签种类：{len(set(raw_chords))} / {len(set(smoothed_chords))}")
print("前 20 帧：", smoothed_chords[:20])


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

img = librosa.display.specshow(
    chroma, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="chroma", ax=axes[0], cmap="Greys_r"
)
axes[0].set_title("Chroma（模板输入）")
axes[0].set_xlabel("")
axes[0].set_ylabel("音级")
fig.colorbar(img, ax=axes[0])

unique_chords = sorted(set(smoothed_chords))
chord_to_int = {label: index for index, label in enumerate(unique_chords)}
int_seq = np.array([chord_to_int[label] for label in smoothed_chords])
axes[1].step(time_chords, int_seq, where="mid", color="0.2", lw=1.5)
axes[1].set_yticks(range(len(unique_chords)))
axes[1].set_yticklabels(unique_chords, fontsize=8)
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("模板标签")
axes[1].set_title("相似度时间平滑 + N 门限 + argmax")
axes[1].set_ylim(-0.5, len(unique_chords) - 0.5)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "chord_recognition_template.png", dpi=600, bbox_inches="tight")
plt.show()

# 对照逐帧 argmax 与相似度中值平滑；两者使用相同 N 门限
unique_compare = sorted(set(raw_chords) | set(smoothed_chords))
compare_to_int = {label: index for index, label in enumerate(unique_compare)}
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].step(
    time_chords, [compare_to_int[label] for label in raw_chords],
    where="mid", color="0.2", lw=1.2, label="逐帧 argmax"
)
axes[1].step(
    time_chords, [compare_to_int[label] for label in smoothed_chords],
    where="mid", color="0.55", lw=1.5, label="相似度中值平滑后 argmax"
)
for ax in axes:
    ax.set_yticks(range(len(unique_compare)))
    ax.set_yticklabels(unique_compare, fontsize=8)
    ax.set_ylabel("标签")
    ax.set_ylim(-0.5, len(unique_compare) - 0.5)
    ax.legend(loc="upper right")
axes[0].set_title("未平滑的逐帧模板输出", loc="left")
axes[1].set_title("相似度沿时间中值平滑后的输出", loc="left")
axes[1].set_xlabel("时间 (s)")
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "chord_recognition_comparison.png", dpi=600, bbox_inches="tight")
plt.show()


## 4.1 Essentia `ChordsDetection`（experimental）

这里调用的是 Essentia 的 `ChordsDetection`：它在 HPCP 序列上用居中的时间窗匹配 major/minor 和弦，并被官方文档标为 experimental。它不是 Chordino，也没有在此接口中运行 HMM。

In [ ]:
# Essentia ChordsDetection：HPCP + 居中时间窗的 major/minor 匹配
chords_detection_available = False
try:
    import essentia.standard as es
    chords_detection_available = True
except ImportError:
    print("Essentia 未安装，跳过 ChordsDetection")

if chords_detection_available:
    window = es.Windowing(type="hann")
    spectrum = es.Spectrum()
    spectral_peaks = es.SpectralPeaks(
        orderBy="magnitude", magnitudeThreshold=0.00001,
        minFrequency=50, maxFrequency=5000, maxPeaks=60,
        sampleRate=SAMPLE_RATE
    )
    hpcp_algo = es.HPCP(size=12, bandPreset=False, sampleRate=SAMPLE_RATE)

    frame_size = 4096
    hop_size = 512
    pcps = []
    for frame in es.FrameGenerator(
        piano_samples, frameSize=frame_size,
        hopSize=hop_size, startFromZero=True
    ):
        spec = spectrum(window(frame))
        freqs, mags = spectral_peaks(spec)
        pcps.append(hpcp_algo(freqs, mags))

    pcps = np.asarray(pcps)
    chords_detection = es.ChordsDetection(
        hopSize=hop_size, sampleRate=SAMPLE_RATE
    )
    essentia_labels, essentia_strength = chords_detection(pcps)
    # FrameGenerator(startFromZero=True) 产生未居中帧；标签时间取分析帧中心。
    time_essentia = librosa.frames_to_time(
        np.arange(len(essentia_labels)), sr=SAMPLE_RATE,
        hop_length=hop_size, n_fft=frame_size
    )

    def essentia_to_std(label):
        """统一质量后缀，并把降号根音规范为模板分支使用的升号拼写。"""
        if label == "N":
            return "N"
        root = label[:-1] if label.endswith("m") else label
        quality = "min" if label.endswith("m") else "maj"
        enharmonic_to_sharp = {
            "Db": "C#", "Eb": "D#", "Gb": "F#",
            "Ab": "G#", "Bb": "A#",
        }
        return f"{enharmonic_to_sharp.get(root, root)}:{quality}"

    essentia_labels_std = [essentia_to_std(label) for label in essentia_labels]
    print(f"ChordsDetection 输出 {len(set(essentia_labels_std))} 种标签")

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    unique_all = sorted(set(smoothed_chords) | set(essentia_labels_std))
    chord_to_int_all = {label: index for index, label in enumerate(unique_all)}

    axes[0].step(
        time_chords, [chord_to_int_all[label] for label in smoothed_chords],
        where="mid", color="0.2", lw=1.5, label="模板法（平滑 + N）"
    )
    axes[0].set_title("模板法输出")
    axes[0].legend(loc="upper right")

    axes[1].step(
        time_essentia, [chord_to_int_all[label] for label in essentia_labels_std],
        where="mid", color="0.55", lw=1.5, label="Essentia ChordsDetection"
    )
    axes[1].set_title("Essentia ChordsDetection 输出（非 Chordino/HMM）")
    axes[1].set_xlabel("时间 (s)")
    axes[1].legend(loc="upper right")

    for ax in axes:
        ax.set_yticks(range(len(unique_all)))
        ax.set_yticklabels(unique_all, fontsize=8)
        ax.set_ylabel("标签")
        ax.set_ylim(-0.5, len(unique_all) - 0.5)

    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_DIR / "chord_recognition_essentia_comparison.png", dpi=600, bbox_inches="tight")
    plt.show()
else:
    print("ChordsDetection 不可用")


## 5. 混音中的人声音高：RMVPE

RMVPE 论文模型使用 deep U-Net + BiGRU、256-bin log-mel、20 ms hop 与 360 维输出，论文默认 voicing 阈值为 0.5。本 Notebook 使用的 `rmvpe-onnx` 0.2.3 是 MIT 社区封装，不是论文官方实现；它实际采用 16 kHz、128 mel 与 10 ms hop，并加载约 345 MB 的 ONNX 权重。

纯人声分轨上的 RMVPE 输出仍是同一模型的估计，不是独立 F0 真值。下图只对照该模型在混音与分轨输入上的输出。


In [ ]:
# 使用 CPU provider 避免不同机器的 CoreML/CUDA provider 差异
rmvpe_available = False
try:
    from rmvpe_onnx import RMVPE
    rmvpe = RMVPE(device="cpu")
    rmvpe_available = True
    model_size_mb = Path(rmvpe.model_path).stat().st_size / (1024 ** 2)
    print(f"RMVPE 社区封装已加载：{rmvpe.model_path}")
    print(f"ONNX 权重大小：{model_size_mb:.1f} MB")
except Exception as exc:
    print(f"RMVPE 加载失败：{exc}")
    rmvpe = None


In [ ]:
# 混音与纯人声分轨：两者都由同一 RMVPE 模型估计
if rmvpe_available and torchcrepe_available:
    mix, _ = librosa.load(
        DATASET_DIR / "xiaohetang_full.wav", sr=SAMPLE_RATE,
        mono=True, offset=30.0, duration=5.0
    )
    vocal, _ = librosa.load(
        DATASET_DIR / "xiaohetang_vox.wav", sr=SAMPLE_RATE,
        mono=True, offset=30.0, duration=5.0
    )

    time_rmvpe, freq_rmvpe, conf_rmvpe, _ = rmvpe.predict(mix, SAMPLE_RATE)
    time_vocal, freq_vocal, conf_vocal, _ = rmvpe.predict(vocal, SAMPLE_RATE)

    audio_tensor = torch.from_numpy(mix).unsqueeze(0).float()
    # 与前面的 torchcrepe 调用相同。
    dither_rng_state = np.random.get_state()
    np.random.seed(202605)
    try:
        f0_mix_crepe, conf_mix_crepe = tc.predict(
            audio_tensor, SAMPLE_RATE, hop_length=160,
            fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"),
            model="tiny", return_periodicity=True, device="cpu"
        )
    finally:
        np.random.set_state(dither_rng_state)
    f0_mix_crepe = f0_mix_crepe.cpu().numpy().flatten()
    conf_mix_crepe = conf_mix_crepe.cpu().numpy().flatten()
    # torchcrepe 默认 pad=True，输出帧中心从输入时刻 0 开始。
    time_mix_crepe = librosa.times_like(
        f0_mix_crepe, sr=SAMPLE_RATE, hop_length=160
    )

    rmvpe_mask = (conf_rmvpe >= 0.5) & np.isfinite(freq_rmvpe) & (freq_rmvpe > 0)
    vocal_mask = (conf_vocal >= 0.5) & np.isfinite(freq_vocal) & (freq_vocal > 0)
    crepe_mask = (conf_mix_crepe >= 0.5) & np.isfinite(f0_mix_crepe) & (f0_mix_crepe > 0)
    print(
        f"torchcrepe 混音：{crepe_mask.sum()} / {len(crepe_mask)} 帧 "
        f"periodicity ≥ 0.5（最大值 {conf_mix_crepe.max():.3f}）"
    )
    print(f"RMVPE 混音：{rmvpe_mask.sum()} / {len(rmvpe_mask)} 帧 confidence ≥ 0.5")
    print(f"RMVPE 纯人声分轨：{vocal_mask.sum()} / {len(vocal_mask)} 帧 confidence ≥ 0.5")

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(time_mix_crepe[crepe_mask], f0_mix_crepe[crepe_mask],
                 color="0.35", lw=1.0, label="torchcrepe 混音（periodicity ≥ 0.5）")
    if not np.any(crepe_mask):
        axes[0].text(
            0.5, 0.5, f"periodicity ≥ 0.5：0 / {len(crepe_mask)} 帧",
            transform=axes[0].transAxes, ha="center", va="center", color="0.35"
        )
    axes[0].set_ylabel("频率 (Hz)")
    axes[0].set_title("torchcrepe：混音输出", loc="left")
    axes[0].legend(loc="upper right")
    axes[0].set_ylim(100, 500)

    axes[1].plot(time_rmvpe[rmvpe_mask], freq_rmvpe[rmvpe_mask],
                 color="0.25", lw=1.2, label="RMVPE 混音（confidence ≥ 0.5）")
    axes[1].plot(time_vocal[vocal_mask], freq_vocal[vocal_mask],
                 color="0.65", lw=1.0, ls="--",
                 label="RMVPE 纯人声分轨（同模型，非真值）")
    axes[1].set_ylabel("频率 (Hz)")
    axes[1].set_xlabel("时间 (s)")
    axes[1].set_title("RMVPE：混音与纯人声分轨的同模型对照", loc="left")
    axes[1].legend(loc="upper right")
    axes[1].set_ylim(100, 500)

    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_DIR / "rmvpe_vs_torchcrepe_mix.png", dpi=600, bbox_inches="tight")
    plt.show()
else:
    print("缺少 RMVPE 或 torchcrepe，跳过对比。")


## 6. 调性估计：Krumhansl--Schmuckler profile

K-S profile 来自 probe-tone 契合度评分实验，不是音频语料的频数统计。下面报告 24 个候选的相关系数排名。

In [ ]:
# Krumhansl--Kessler probe-tone profiles（以 C 为根音）
KS_MAJOR = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
KS_MINOR = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

def estimate_key_ks(chroma_mean):
    chroma_mean = np.asarray(chroma_mean, dtype=float)
    if chroma_mean.shape != (12,):
        raise ValueError("chroma_mean 必须是 12 维向量")
    if not np.all(np.isfinite(chroma_mean)) or np.std(chroma_mean) <= 1e-12:
        raise ValueError("静音、常量或非有限 chroma 无法计算有意义的 Pearson 相关")
    chroma_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    correlations, labels = [], []
    for shift in range(12):
        correlations.extend([
            np.corrcoef(chroma_mean, np.roll(KS_MAJOR, shift))[0, 1],
            np.corrcoef(chroma_mean, np.roll(KS_MINOR, shift))[0, 1],
        ])
        labels.extend([f"{chroma_names[shift]} 大调", f"{chroma_names[shift]} 小调"])
    correlations = np.asarray(correlations)
    best_idx = int(np.argmax(correlations))
    return labels[best_idx], correlations, labels

song_full, sr = librosa.load(
    DATASET_DIR / "xiaohetang_full.wav", sr=SAMPLE_RATE,
    mono=True, offset=30.0, duration=20.0
)
chroma_song = librosa.feature.chroma_cqt(
    y=song_full, sr=SAMPLE_RATE, hop_length=512
)
chroma_mean = np.mean(chroma_song, axis=1)

best_key, all_corrs, all_labels = estimate_key_ks(chroma_mean)
ranking = np.argsort(all_corrs)[::-1]
print("K-S profile 排名前三（无独立调性真值）：")
for rank, index in enumerate(ranking[:3], start=1):
    print(f"  {rank}. {all_labels[index]}: r={all_corrs[index]:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
corr_matrix = all_corrs.reshape(12, 2).T
im = ax.imshow(corr_matrix, aspect="auto", cmap="Greys", vmin=-0.5, vmax=1.0)
if np.min(corr_matrix) < 0 < np.max(corr_matrix):
    ax.contour(corr_matrix, levels=[0], colors="white", linewidths=1.2, linestyles="--")
ax.set_xticks(range(12))
ax.set_xticklabels(["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["大调", "小调"])
ax.set_title(f"K-S profile 相关：最高候选 = {best_key}（r={np.max(all_corrs):.3f}）")
fig.colorbar(im, ax=ax, label="皮尔逊相关系数 r")

best_idx = int(ranking[0])
ax.plot(best_idx // 2, best_idx % 2, "k*", markersize=20)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "ks_key_estimation.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
print("输出目录中与本 Notebook 对应的现有图像文件：")
for prefix in ["pitch_", "basic_pitch", "chord_", "ks_key", "rmvpe_", "multipitch_"]:
    for f in OUTPUT_FIG_DIR.glob(f"{prefix}*.png"):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} {size_kb:8.1f} KB")